In [2]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio as rio
from shapely.geometry import box
from rasterio.features import rasterize
from rasterio.transform import Affine
from tqdm import tqdm
import math, os
from matplotlib import pyplot as plt
from pathlib import Path

In [ ]:
ROOT = Path.cwd().parent
DATA = ROOT / 'data'
RESULTS = ROOT / 'results'
(RESULTS / 'species-range-rasters' / 'reptiles').mkdir(parents=True, exist_ok=True)

In [3]:
country_raster_1km_ = rio.open(RESULTS / 'country-rasters' / 'country-raster-eck4-1km.tif')

### reptiles

In [4]:
reptiles_1 = gpd.read_file(DATA / 'species-range-maps' / 'REPTILES' / 'REPTILES_PART1.shp')

In [5]:
reptiles_2 = gpd.read_file(DATA / 'species-range-maps' / 'REPTILES' / 'REPTILES_PART2.shp')

In [6]:
reptiles_ = pd.concat([reptiles_1, reptiles_2]).reset_index(drop=True)

In [7]:
reptiles = reptiles_[(reptiles_['presence'] == (1 or 2 or 3)) & (reptiles_['origin'] == (1 or 2 or 6)) & (reptiles_['terrestria'] == 'true')]

In [8]:
reptiles_eck4 = reptiles.to_crs('ESRI:54012')

In [9]:
species_names = reptiles_eck4['sci_name'].unique().tolist()
species_category = [reptiles_eck4[reptiles_eck4['sci_name'] == species]['category'].values[0] for species in species_names]

range_index_reptiles = pd.DataFrame([], columns=['name', 'row_min', 'row_max', 'col_min', 'col_max'])
range_index_reptiles['name'] = species_names
range_index_reptiles['category'] = species_category

In [10]:
for i in tqdm(range_index_reptiles.index):
    a_species = range_index_reptiles.loc[i, 'name']
    try:
        # print(a_species)
        a_species_range = reptiles_eck4[reptiles_eck4['sci_name'] == a_species].geometry 
        a_species_polygons = [(a_geometry, 1) for a_geometry in a_species_range]
        reprojected_raster = rasterize(
            shapes=a_species_polygons, 
            out_shape=country_raster_1km_.shape, 
            transform=country_raster_1km_.transform,
            all_touched=True,
            fill=0, 
            dtype=np.uint8,
        )
        rows, cols = np.where(reprojected_raster == 1)
        row_min, row_max = rows.min(), rows.max()
        col_min, col_max = cols.min(), cols.max()
        the_slice = reprojected_raster[row_min:row_max+1, col_min:col_max+1]

        output_path = RESULTS / 'species-range-rasters' / 'reptiles' / f'{a_species}.tif'
        new_transform = country_raster_1km_.transform * Affine.translation(col_min, row_min)

        with rio.open(
            output_path,
            'w',
            driver='GTiff',
            height=the_slice.shape[0],
            width=the_slice.shape[1],
            count=1,
            dtype=the_slice.dtype, 
            crs=country_raster_1km_.crs,
            transform=new_transform,
            compress='lzw',
        ) as dst:
            dst.write(the_slice, 1)

        range_index_reptiles.loc[i, ['row_min', 'row_max', 'col_min', 'col_max']] = [row_min, row_max, col_min, col_max]
        range_index_reptiles.to_csv(RESULTS / 'species-range-rasters' / 'range-index-reptiles.csv', index=False)
        
    except:
        pass


100%|██████████| 9870/9870 [3:37:13<00:00,  1.32s/it]  
